# Proyecto Final
## Clasificación automática y descubrimiento de sub-temas en quejas de consumidores financieros
**Camino B:** Supervisado + NLP + No supervisado

Dataset: CFPB Consumer Complaint Database (2011–2019)

## 0. Dataset y alcance

- **Fuente:** CFPB Consumer Complaint Database (`rows.csv`).
- **Decisión clave:** `Consumer complaint narrative` está vacía en ~68–70% de filas.
  Para NLP usamos solo filas con narrativa (documentamos cuántas quedan).
- **Product:** unificamos categorías renombradas y nos quedamos con el **Top 7 + Otros**.
- **Objetivo:** enrutamiento automático de quejas + detección de sub-temas emergentes.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import geopandas as gpd

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, f1_score, silhouette_score,
)
from sklearn.model_selection import train_test_split

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
DATA_PATH = Path('rows.csv')
US_STATES_URL = 'https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json'

## 1. Análisis de datos y metadatos (5%)

In [ ]:
DATA_DICTIONARY = {
    'Date received': 'Fecha en que el CFPB recibió la queja.',
    'Product': 'Categoría principal del producto o servicio financiero.',
    'Sub-product': 'Subcategoría del producto.',
    'Issue': 'Problema reportado por el consumidor.',
    'Sub-issue': 'Detalle del problema dentro del Issue.',
    'Consumer complaint narrative': 'Texto libre de la queja (puede estar vacío).',
    'Company public response': 'Respuesta pública de la empresa (puede estar vacía).',
    'Company': 'Institución financiera señalada.',
    'State': 'Estado de EE.UU. del consumidor.',
    'ZIP code': 'Código postal del consumidor.',
}

usecols = list(DATA_DICTIONARY.keys())
df_raw = pd.read_csv(DATA_PATH, usecols=usecols, low_memory=False)
df_raw['Date received'] = pd.to_datetime(df_raw['Date received'], errors='coerce')
df = df_raw.copy()

print('Shape:', df.shape)
meta = pd.DataFrame({
    'columna': df.columns,
    'tipo': df.dtypes.astype(str).values,
    'nulos_%': (df.isna().mean() * 100).round(2).values,
    'descripcion': [DATA_DICTIONARY[c] for c in df.columns],
})
meta

In [ ]:
plt.figure(figsize=(10, 4))
sns.heatmap(df.isna(), cbar=False, cmap='Reds')
plt.title('Mapa de valores nulos')
plt.tight_layout()
plt.show()

print('Duplicados exactos:', df.duplicated().sum())
print('Rango de fechas:', df['Date received'].min(), '→', df['Date received'].max())
print('Narrativas no nulas:', df['Consumer complaint narrative'].notna().sum(),
      f"({df['Consumer complaint narrative'].notna().mean()*100:.1f}%)")

**Interpretación metadatos:** La narrativa y la respuesta pública concentran la mayoría de nulos. El EDA agregado puede usar todo el dataset; el NLP solo filas con texto.

### 1.5 Preparación y limpieza de datos (visible)

El CFPB renombró categorías de `Product` a lo largo de los años. Unificamos variantes equivalentes antes de modelar.

In [ ]:
PRODUCT_UNIFICATION = {
    'Credit reporting, credit repair services, or other personal consumer reports': 'Credit reporting',
    'Credit card or prepaid card': 'Credit card',
    'Checking or savings account': 'Bank account or service',
}

TOP_PRODUCTS = [
    'Credit reporting', 'Mortgage', 'Debt collection', 'Credit card',
    'Bank account or service', 'Student loan', 'Consumer Loan',
]

print('Product ORIGINAL (top 10):')
display(df_raw['Product'].value_counts().head(10))

df['Product_unified'] = df['Product'].replace(PRODUCT_UNIFICATION)
df.loc[~df['Product_unified'].isin(TOP_PRODUCTS), 'Product_unified'] = 'Otros'

print('\nProduct UNIFICADO (top 10):')
display(df['Product_unified'].value_counts().head(10))

**Por qué unificamos:** *Credit reporting* aparecía fragmentado en dos etiquetas históricas; lo mismo con *Credit card* y cuentas bancarias. Sin esto, el clasificador vería clases artificiales.

In [ ]:
narr = df.loc[df['Consumer complaint narrative'].notna(), 'Consumer complaint narrative']
pct_xxxx = narr.str.contains(r'X{2,}', regex=True, na=False).mean() * 100
print(f'{pct_xxxx:.1f}% de narrativas contienen enmascaramiento XXXX (datos personales redactados)')

ejemplos_xxxx = narr[narr.str.contains('XXXX', na=False)].head(3)
for i, txt in enumerate(ejemplos_xxxx, 1):
    print(f'\n--- Ejemplo {i} ---\n{txt[:450]}...')

**Detalle del dataset:** Las narrativas CFPB enmascaran nombres, montos y fechas con `XXXX`. Nuestra limpieza trata esto explícitamente (no es un regex genérico).

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'\b[x]{2,}\b', ' redacted ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

muestras = df[df['Consumer complaint narrative'].notna()].sample(4, random_state=42)
comparacion = pd.DataFrame({
    'original': muestras['Consumer complaint narrative'].str[:350].values,
    'despues_clean_text': muestras['Consumer complaint narrative'].map(clean_text).str[:350].values,
})
comparacion

In [ ]:
df['has_narrative'] = df['Consumer complaint narrative'].notna()
df['year'] = df['Date received'].dt.year
df['quarter'] = df['Date received'].dt.to_period('Q').astype(str)

df[['Date received', 'Product', 'Product_unified', 'has_narrative', 'year', 'quarter']].head(8)

In [ ]:
nlp_df = df[df['has_narrative']].copy()
print(f'Filas con narrativa: {len(nlp_df):,} de {len(df):,} ({len(nlp_df)/len(df)*100:.1f}%)')
print('\nDistribución por Product (antes de muestreo):')
print(nlp_df['Product_unified'].value_counts())

SAMPLE_N = 100_000
if len(nlp_df) > SAMPLE_N:
    parts = []
    for _, group in nlp_df.groupby('Product_unified'):
        n = max(500, int(SAMPLE_N * len(group) / len(nlp_df)))
        parts.append(group.sample(min(len(group), n), random_state=42))
    nlp_df = pd.concat(parts).reset_index(drop=True)
    print(f'\nMuestra estratificada para NLP: {len(nlp_df):,} filas')
    print(nlp_df['Product_unified'].value_counts())

## 2. EDA — Exploratory Data Analysis (10%)

In [ ]:
prod = df['Product_unified'].value_counts().sort_values()
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(prod.index, prod.values, color='steelblue')
ax.set_title('Distribución de quejas por Product (unificado)')
ax.set_xlabel('Número de quejas')
ax.bar_label(bars, fmt='{:,.0f}', padding=3, fontsize=9)
ax.grid(axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

**Interpretación:** *Credit reporting* y *Mortgage* dominan el volumen → prioridad para monitoreo regulatorio.

In [ ]:
ts = df.groupby('quarter').size()
ts.plot(figsize=(12, 4), marker='o')
plt.title('Serie de tiempo — quejas por trimestre')
plt.ylabel('Quejas')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**Interpretación:** El volumen crece hacia 2016–2018, coherente con mayor digitalización y visibilidad del CFPB.

In [ ]:
top_co = df['Company'].value_counts().head(15)
ax = top_co.plot(kind='barh', figsize=(10, 6), color='coral')
plt.title('Top 15 Company por volumen de quejas')
plt.xlabel('Quejas')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
top5 = df['Company'].value_counts().head(5).index
comp_prod = (
    df[df['Company'].isin(top5)]
    .groupby(['Company', 'Product_unified'])
    .size()
    .unstack(fill_value=0)
)
fig, ax = plt.subplots(figsize=(13, 6))
comp_prod.plot(kind='bar', stacked=True, ax=ax, colormap='tab20', width=0.75)
ax.set_title('Composición por Product — Top 5 empresas')
ax.set_ylabel('Quejas')
ax.set_xlabel('Company')
plt.xticks(rotation=25, ha='right')
ax.legend(title='Product', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

**Interpretación:** Las grandes instituciones no comparten el mismo mix de Product → el clasificador debe apoyarse en el texto, no solo en la empresa.

In [ ]:
STATE_ABBR_TO_NAME = {
    'AL': 'Alabama', 'AK': 'Alaska', 'AZ': 'Arizona', 'AR': 'Arkansas', 'CA': 'California',
    'CO': 'Colorado', 'CT': 'Connecticut', 'DE': 'Delaware', 'FL': 'Florida', 'GA': 'Georgia',
    'HI': 'Hawaii', 'ID': 'Idaho', 'IL': 'Illinois', 'IN': 'Indiana', 'IA': 'Iowa',
    'KS': 'Kansas', 'KY': 'Kentucky', 'LA': 'Louisiana', 'ME': 'Maine', 'MD': 'Maryland',
    'MA': 'Massachusetts', 'MI': 'Michigan', 'MN': 'Minnesota', 'MS': 'Mississippi', 'MO': 'Missouri',
    'MT': 'Montana', 'NE': 'Nebraska', 'NV': 'Nevada', 'NH': 'New Hampshire', 'NJ': 'New Jersey',
    'NM': 'New Mexico', 'NY': 'New York', 'NC': 'North Carolina', 'ND': 'North Dakota', 'OH': 'Ohio',
    'OK': 'Oklahoma', 'OR': 'Oregon', 'PA': 'Pennsylvania', 'RI': 'Rhode Island', 'SC': 'South Carolina',
    'SD': 'South Dakota', 'TN': 'Tennessee', 'TX': 'Texas', 'UT': 'Utah', 'VT': 'Vermont',
    'VA': 'Virginia', 'WA': 'Washington', 'WV': 'West Virginia', 'WI': 'Wisconsin', 'WY': 'Wyoming',
    'DC': 'District of Columbia',
}

states = gpd.read_file(US_STATES_URL)
state_counts = df.groupby('State').size().reset_index(name='quejas')
state_counts['name'] = state_counts['State'].map(STATE_ABBR_TO_NAME)
merged = states.merge(state_counts, on='name', how='left')
merged['quejas'] = merged['quejas'].fillna(0)

from matplotlib.colors import Normalize
from matplotlib import cm

fig, ax = plt.subplots(figsize=(12, 7))
merged.plot(column='quejas', ax=ax, cmap='YlOrRd', edgecolor='white', linewidth=0.2, legend=False)
norm = Normalize(vmin=merged['quejas'].min(), vmax=merged['quejas'].max())
sm = cm.ScalarMappable(cmap='YlOrRd', norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, fraction=0.025, pad=0.02)
cbar.set_label('Número de quejas')
ax.set_title('Mapa coroplético — quejas por estado')
ax.axis('off')
plt.tight_layout()
plt.show()

**Interpretación:** CA, FL, TX y NY concentran gran parte del volumen, alineado con población y actividad financiera.

In [ ]:
top_product = df['Product_unified'].value_counts().idxmax()
issue_top = df[df['Product_unified'] == top_product]['Issue'].value_counts().head(12)
issue_top.plot(kind='barh', figsize=(10, 6), color='teal')
plt.title(f'Top Issue dentro de {top_product}')
plt.tight_layout()
plt.show()

In [ ]:
narr_by_prod = df.groupby('Product_unified')['has_narrative'].mean().sort_values(ascending=False) * 100
narr_by_prod.plot(kind='barh', figsize=(10, 5), color='purple')
plt.title('% de quejas con narrativa por Product')
plt.xlabel('% con narrativa')
plt.tight_layout()
plt.show()

narr_time = df.groupby('year')['has_narrative'].mean() * 100
narr_time.plot(figsize=(10, 4), marker='o', color='purple')
plt.title('% de quejas con narrativa por año')
plt.ylabel('%')
plt.tight_layout()
plt.show()

**Interpretación:** El porcentaje con narrativa varía por Product y aumenta en años recientes, reforzando la viabilidad del módulo NLP sobre un subconjunto creciente.

## 3. Propuesta de negocio (5%)

**Problema:** El triage manual de quejas es lento y costoso. Dentro de categorías grandes (p. ej. *Credit reporting*) pueden existir sub-problemas emergentes no capturados a tiempo por una taxonomía fija (*Sub-issue*).

**Propuesta:**
1. Clasificador automático que enrute cada queja nueva a *Product* en tiempo real a partir del texto libre.
2. Módulo no supervisado de sub-temas dentro del producto más grande, como alerta temprana de patrones nuevos.

**Beneficiarios:** CFPB/reguladores, bancos y aseguradoras, software de atención al cliente financiero.

**Métrica de éxito:** Reducción del tiempo de enrutamiento manual (40–60%) y detección de sub-temas emergentes semanas antes que el proceso manual.

## 4. Implementación de Ciencia de Datos (10%)

### 4a. NLP + Supervisado — Clasificación de Product

In [ ]:
nlp_model = nlp_df[nlp_df['Product_unified'] != 'Otros'].copy()
nlp_model['text_clean'] = nlp_model['Consumer complaint narrative'].map(clean_text)
print('Filas para modelado:', len(nlp_model))
nlp_model['Product_unified'].value_counts()

**Hiperparámetros TF-IDF:** Probamos `max_features` 2000 vs 5000 (con `ngram_range=(1,2)` y `min_df=5` para ignorar términos muy raros).

In [ ]:
X_text = nlp_model['text_clean']
y = nlp_model['Product_unified']

for max_feat in [2000, 5000]:
    vec_tmp = TfidfVectorizer(max_features=max_feat, ngram_range=(1, 2), min_df=5)
    X_tmp = vec_tmp.fit_transform(X_text)
    X_tr, X_te, y_tr, y_te = train_test_split(X_tmp, y, test_size=0.2, random_state=42, stratify=y)
    clf_tmp = LogisticRegression(max_iter=1000, class_weight='balanced', n_jobs=-1)
    clf_tmp.fit(X_tr, y_tr)
    f1 = f1_score(y_te, clf_tmp.predict(X_te), average='macro')
    print(f'max_features={max_feat} → vocabulario={X_tmp.shape[1]:,} | F1 macro={f1:.4f}')

Elegimos **max_features=5000** porque captura más n-gramas de productos financieros con una mejora modesta pero estable en F1 macro.

In [ ]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=5)
X = tfidf.fit_transform(nlp_model['text_clean'])
y = nlp_model['Product_unified']

idx = np.arange(len(nlp_model))
idx_train, idx_test = train_test_split(
    idx, test_size=0.2, random_state=42, stratify=y
)
X_train, X_test = X[idx_train], X[idx_test]
y_train, y_test = y.iloc[idx_train], y.iloc[idx_test]

log_reg = LogisticRegression(max_iter=2000, class_weight='balanced', n_jobs=-1)
log_reg.fit(X_train, y_train)
y_pred = log_reg.predict(X_test)

acc_lr = accuracy_score(y_test, y_pred)
f1_lr = f1_score(y_test, y_pred, average='macro')
print(f'Logistic Regression — Accuracy: {acc_lr:.4f} | F1 macro: {f1_lr:.4f}')
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=log_reg.classes_)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=False, cmap='Blues', xticklabels=log_reg.classes_, yticklabels=log_reg.classes_)
plt.title('Matriz de confusión — Logistic Regression')
plt.xlabel('Predicho')
plt.ylabel('Real')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)
acc_dummy = accuracy_score(y_test, y_pred_dummy)
f1_dummy = f1_score(y_test, y_pred_dummy, average='macro')
print(f'Baseline (clase mayoritaria) — Accuracy: {acc_dummy:.4f} | F1 macro: {f1_dummy:.4f}')
print(f'Logistic Regression — Accuracy: {acc_lr:.4f} | F1 macro: {f1_lr:.4f}')
print(f'Lift vs baseline: +{(acc_lr - acc_dummy)*100:.1f} pp accuracy, +{(f1_lr - f1_dummy)*100:.1f} pp F1 macro')

In [ ]:
report_dict = classification_report(y_test, y_pred, output_dict=True)
f1_por_clase = {k: round(v['f1-score'], 3) for k, v in report_dict.items()
                if k not in ('accuracy', 'macro avg', 'weighted avg')}
f1_series = pd.Series(f1_por_clase).sort_values()
f1_series.plot(kind='barh', figsize=(10, 5), color='seagreen')
plt.title('F1 por clase — Logistic Regression')
plt.xlabel('F1-score')
plt.tight_layout()
plt.show()
print('Clases más difíciles:', f1_series.head(2).to_dict())
print('Clases más fáciles:', f1_series.tail(2).to_dict())

In [ ]:
feature_names = tfidf.get_feature_names_out()
term_rows = []
for idx, label in enumerate(log_reg.classes_):
    coefs = log_reg.coef_[idx]
    top_idx = np.argsort(coefs)[-8:][::-1]
    for rank, ti in enumerate(top_idx, 1):
        term_rows.append({'Product': label, 'rank': rank, 'term': feature_names[ti], 'coef': coefs[ti]})
terms = pd.DataFrame(term_rows)
for product in log_reg.classes_[:3]:
    top = terms[terms['Product'] == product].head(8)
    print(f"\n{product}: {', '.join(top['term'].tolist())}")

**Interpretación supervisada (con cifras):** Revisa los valores impresos de **accuracy** y **F1 macro** en las celdas anteriores. El modelo supera claramente el baseline de clase mayoritaria. Las clases con menor F1 suelen ser las de menor volumen (*Student loan*, *Consumer Loan*). Las palabras top por coeficiente son interpretables para negocio.

In [ ]:
errors_df = pd.DataFrame({
    'texto': nlp_model.iloc[idx_test]['Consumer complaint narrative'].values,
    'real': y_test.values,
    'predicho': y_pred,
})
confundidos = errors_df[errors_df['real'] != errors_df['predicho']]
print(f'Errores en test: {len(confundidos):,} de {len(errors_df):,} ({len(confundidos)/len(errors_df)*100:.1f}%)')

for i, row in confundidos.sample(min(3, len(confundidos)), random_state=42).iterrows():
    print(f"\nReal: {row['real']} | Predicho: {row['predicho']}")
    print(row['texto'][:450])
    print('-' * 70)

**Error analysis:** Los casos mal clasificados suelen tener vocabulario compartido entre productos (p. ej. *loan* aparece en Mortgage y Student loan) o narrativas muy cortas tras la limpieza.

### 4b. No supervisado — Sub-temas en Credit reporting

In [ ]:
CLUSTER_PRODUCT = 'Credit reporting'
credit = df[(df['Product_unified'] == CLUSTER_PRODUCT) & df['has_narrative']].copy()
if len(credit) > 25000:
    credit = credit.sample(25000, random_state=42)
credit['text_clean'] = credit['Consumer complaint narrative'].map(clean_text)
print('Muestra clustering:', len(credit))

In [ ]:
tfidf_cl = TfidfVectorizer(max_features=3000, ngram_range=(1, 2), min_df=5)
X_cl = tfidf_cl.fit_transform(credit['text_clean'])

scores = []
K_range = range(3, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_cl)
    scores.append(silhouette_score(X_cl, labels))

plt.figure(figsize=(8, 4))
plt.plot(list(K_range), scores, marker='o')
plt.title('Silhouette score vs k')
plt.xlabel('k')
plt.ylabel('Silhouette')
plt.tight_layout()
plt.show()

best_k = list(K_range)[int(np.argmax(scores))]
print('Mejor k:', best_k, '| silhouette:', round(max(scores), 4))

In [ ]:
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_cl)

cl_terms = tfidf_cl.get_feature_names_out()
cluster_terms = {}
for i, center in enumerate(kmeans.cluster_centers_):
    top_idx = np.argsort(center)[-10:][::-1]
    cluster_terms[i] = [cl_terms[j] for j in top_idx]

pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_cl.toarray())

plt.figure(figsize=(9, 6))
scatter = plt.scatter(coords[:, 0], coords[:, 1], c=cluster_labels, cmap='tab10', alpha=0.5, s=8)
plt.title('Clusters de sub-temas — Credit reporting (PCA 2D)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.colorbar(scatter, label='Cluster')
plt.tight_layout()
plt.show()

cluster_names = {}
for cid, words in cluster_terms.items():
    nombre = ', '.join(words[:5])
    cluster_names[cid] = nombre
    print(f'Cluster {cid} — {nombre}')

In [ ]:
compare = pd.DataFrame({
    'cluster': cluster_labels,
    'nombre_cluster': [cluster_names[c] for c in cluster_labels],
    'Sub-issue': credit['Sub-issue'].fillna('Sin sub-issue').values,
})

print('Top Sub-issue por cluster (con nombre interpretado):')
for cid in sorted(cluster_names.keys()):
    sub = compare[compare['cluster'] == cid]['Sub-issue'].value_counts().head(3)
    print(f"\nCluster {cid} [{cluster_names[cid][:60]}...]")
    print(sub.to_string())

**Interpretación no supervisada:** Los clusters recuperan patrones semánticos alineados con Sub-issue oficiales (p. ej. identity theft, incorrect information on report), pero también agrupan quejas con etiquetas dispersas → potencial para detectar temas emergentes no catalogados.

## 5. Data Storytelling y visualización (5%)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df['Product_unified'].value_counts().head(6).plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Hallazgo 1: Productos dominantes')
axes[0].tick_params(axis='x', rotation=30)

f1_series.plot(kind='barh', ax=axes[1], color='seagreen')
axes[1].set_title('Hallazgo 2: F1 por Product (clasificador)')
axes[1].set_xlabel('F1-score')
plt.suptitle('Resumen ejecutivo visual — CFPB Complaints', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**Narrativa:** El volumen se concentra en crédito e hipotecas. El clasificador de texto supera el baseline de clase mayoritaria (ver celda de comparación). El clustering complementa detectando sub-temas dentro de *Credit reporting*.

## 6. Conclusiones y aportaciones (5%)

**¿Se sostiene la hipótesis?** Sí, parcialmente: el clasificador supervisado enruta quejas con F1 macro claramente superior al baseline, y el clustering revela sub-estructura útil dentro de *Credit reporting*.

**Limitaciones:** ~70% sin narrativa; desbalance de clases; datos hasta 2019; TF-IDF no captura contexto profundo.

**Próximos pasos:** Datos recientes, embeddings/BERT, integrar Sub-product como feature, despliegue en tiempo real.

**Aportación accionable:** Pipeline documentado celda por celda para enrutamiento automático + alertas tempranas de sub-temas.